# Processamento de Imagem X-ray

Notebook de apoio para a atividade de Métodos Numéricos com leitura, visualização, detecção de bordas e aplicação de máscaras em imagens de raio-x.

## 1. Importações

In [ ]:
import os
import numpy as np
import imageio.v3 as imageio
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"


## 2. Examinar uma imagem X-ray

In [ ]:
DIR = "tutorial-x-ray-image-processing"

xray_image = imageio.imread(os.path.join(DIR, "00000011_001.png"))

print("Shape:", xray_image.shape)
print("Dtype:", xray_image.dtype)

fig, ax = plt.subplots()
ax.imshow(xray_image, cmap="gray")
ax.set_axis_off()
ax.set_title("X-ray original")
plt.show()


## 3. Combinar 9 imagens em um array 3D

In [ ]:
num_imgs = 9

combined_xray_images_1 = np.array(
    [imageio.imread(os.path.join(DIR, f"00000011_00{i}.png")) for i in range(num_imgs)]
)

print("Shape do array combinado:", combined_xray_images_1.shape)

fig, axes = plt.subplots(nrows=1, ncols=num_imgs, figsize=(30, 4))
for img, ax in zip(combined_xray_images_1, axes):
    ax.imshow(img, cmap="gray")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Criação do GIF
GIF_PATH = os.path.join(DIR, "xray_image.gif")
imageio.mimwrite(GIF_PATH, combined_xray_images_1, format=".gif", duration=1000)

print("GIF salvo em:", GIF_PATH)


## 4. Filtro Laplaciano-Gaussiano

In [ ]:
xray_image_laplace_gaussian = ndimage.gaussian_laplace(xray_image, sigma=1)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Laplacian-Gaussian")
axes[1].imshow(xray_image_laplace_gaussian, cmap="gray")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Gradiente Gaussiano

In [ ]:
x_ray_image_gaussian_gradient = ndimage.gaussian_gradient_magnitude(xray_image, sigma=2)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Gaussian Gradient Magnitude")
axes[1].imshow(x_ray_image_gaussian_gradient, cmap="gray")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Filtro de Sobel

In [ ]:
x_sobel = ndimage.sobel(xray_image, axis=0)
y_sobel = ndimage.sobel(xray_image, axis=1)

xray_image_sobel = np.hypot(x_sobel, y_sobel)
xray_image_sobel *= 255.0 / np.max(xray_image_sobel)
xray_image_sobel = xray_image_sobel.astype("float32")

print("Dtype Sobel:", xray_image_sobel.dtype)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Sobel - grayscale")
axes[1].imshow(xray_image_sobel, cmap="gray")
axes[2].set_title("Sobel - CMRmap")
axes[2].imshow(xray_image_sobel, cmap="CMRmap")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 7. Versão funcional do exemplo "Canny" do tutorial

In [ ]:
smoothed = ndimage.gaussian_filter(xray_image, sigma=1)

x_prewitt = ndimage.prewitt(smoothed, axis=0)
y_prewitt = ndimage.prewitt(smoothed, axis=1)

xray_image_canny = np.hypot(x_prewitt, y_prewitt)
xray_image_canny *= 255.0 / np.max(xray_image_canny)

print("Dtype Canny-like:", xray_image_canny.dtype)

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 5))
axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Edges - prism")
axes[1].imshow(xray_image_canny, cmap="prism")
axes[2].set_title("Edges - nipy_spectral")
axes[2].imshow(xray_image_canny, cmap="nipy_spectral")
axes[3].set_title("Edges - terrain")
axes[3].imshow(xray_image_canny, cmap="terrain")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 8. Estatísticas e máscaras com `np.where()`

In [ ]:
print("Dtype:", xray_image.dtype)
print("Mínimo:", np.min(xray_image))
print("Máximo:", np.max(xray_image))
print("Média:", np.mean(xray_image))
print("Mediana:", np.median(xray_image))

pixel_intensity_distribution = ndimage.histogram(
    xray_image,
    min=np.min(xray_image),
    max=np.max(xray_image),
    bins=256
)

fig, ax = plt.subplots()
ax.plot(pixel_intensity_distribution)
ax.set_title("Pixel intensity distribution")
ax.set_xlabel("Intensity")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

xray_image_mask_noisy = np.where(xray_image > 150, xray_image, 0)

fig, ax = plt.subplots()
ax.imshow(xray_image_mask_noisy, cmap="gray")
ax.set_axis_off()
ax.set_title("Mask > 150 (noisy)")
plt.show()

xray_image_mask_less_noisy = np.where(xray_image > 150, 1, 0)

fig, ax = plt.subplots()
ax.imshow(xray_image_mask_less_noisy, cmap="gray")
ax.set_axis_off()
ax.set_title("Mask > 150 (binary)")
plt.show()


## 9. Comparação final dos resultados

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=8, figsize=(28, 4))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")

axes[1].set_title("Laplace-Gaussian")
axes[1].imshow(xray_image_laplace_gaussian, cmap="gray")

axes[2].set_title("Gaussian gradient")
axes[2].imshow(x_ray_image_gaussian_gradient, cmap="gray")

axes[3].set_title("Sobel")
axes[3].imshow(xray_image_sobel, cmap="gray")

axes[4].set_title("Sobel - hot")
axes[4].imshow(xray_image_sobel, cmap="hot")

axes[5].set_title("Edges - prism")
axes[5].imshow(xray_image_canny, cmap="prism")

axes[6].set_title("Mask > 150")
axes[6].imshow(xray_image_mask_noisy, cmap="gray")

axes[7].set_title("Binary mask")
axes[7].imshow(xray_image_mask_less_noisy, cmap="gray")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 10. Conclusão

A atividade demonstrou como carregar imagens médicas, empilhar várias imagens em um array multidimensional e aplicar filtros para realce de bordas e segmentação simples com máscaras.